# Memory AI Lab — Boundary Detector Training

**Workflow :** éditer le code dans VS Code → `git push` → ouvrir ce notebook dans [colab.research.google.com](https://colab.research.google.com)

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

## Ce que fait ce notebook

```
Stage 1 — Boundary Detector
   Input  : embeddings mE5-base (768d) + gap temporel
   Modèle : BoundaryMLP 770 → 128 → 32 → 1
   Output : P(frontière) pour chaque message
   Durée  : ~2 min CPU | ~20 sec GPU
```

**Données requises sur Google Drive (`memory_ai_data/`) :**
```
group_anon.txt
group_gold_tune.json
group_gold_test.json
```

Le modèle entraîné est sauvegardé dans `memory_ai_data/boundary_detector.pt`.

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!python -m spacy download fr_core_news_sm -q
print('✓ OK')
print()
print('⚠️  Si première exécution : Exécution > Redémarrer la session,')
print('   puis relancer à partir de la cellule 3.')

In [ ]:
# ── CELLULE 3 : Google Drive + Copie locale (évite les coupures Drive) ───
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/memory_ai_data'
LOCAL_DIR = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

FILES_TO_COPY = [
    'group_anon.txt',
    'group_gold_tune.json',
    'group_gold_test.json',
    'group_embeddings_me5.npy',   # optionnel — cache embeddings
]
for fname in FILES_TO_COPY:
    src = f'{DRIVE_DIR}/{fname}'
    dst = f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        print(f'  Copie {fname} ...', end=' ', flush=True)
        shutil.copy2(src, dst)
        print('✓')
    elif os.path.exists(dst):
        print(f'  {fname} déjà en local ✓')
    else:
        print(f'  {fname} absent sur Drive (ignoré)')

DATA_DIR = LOCAL_DIR
print(f'\n✓ DATA_DIR = {DATA_DIR}')

In [ ]:
# ── CELLULE 4 : Parse + Embeddings mE5-base (GPU + cache) ─────────────────
import numpy as np
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from parsers.whatsapp_parser import parse_whatsapp_chat

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

# mE5-base — multilingual, 768d, handles FR/EN code-switching
MODEL_NAME  = 'intfloat/multilingual-e5-base'
PREFIX      = 'passage: '   # requis par mE5 pour les documents
EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings_me5.npy'

all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
texts = [PREFIX + a.content for a in all_artifacts]
print(f'[1/2] {len(texts)} messages parsés')

if EMBED_CACHE.exists():
    all_embeddings = np.load(EMBED_CACHE)
    assert len(all_embeddings) == len(texts), 'Cache périmé — supprimer group_embeddings_me5.npy'
    print('[2/2] Embeddings chargés depuis cache')
else:
    print(f'[2/2] Calcul sur {device} (mE5-base 768d) ...')
    model = SentenceTransformer(MODEL_NAME, device=device)
    all_embeddings = model.encode(
        texts, batch_size=256, show_progress_bar=True,
        device=device, convert_to_numpy=True
    ).astype(np.float32)
    np.save(EMBED_CACHE, all_embeddings)
    print(f'      Sauvegardé → {EMBED_CACHE}')

print(f'      Shape : {all_embeddings.shape}  (attendu : (n, 768))')

In [ ]:
# ── CELLULE 5 : Charger tune_early (entraînement boundary detector) ────────
# Protocole B : le boundary detector est entraîné sur la première fraction
# du tune uniquement. La seconde fraction (tune_late) est réservée à Optuna.
# → Aucune fuite de données entre Stage 1 et l'optimisation de Stage 2.
#
# TUNE_SPLIT doit être identique dans 01_eval_ari.ipynb.
TUNE_SPLIT = 0.65   # 65% tune_early pour boundary detector, 35% tune_late pour Optuna

import json

def load_split(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    n = len(data['artifacts'])
    y_true = [None] * n
    for ep in data['episodes']:
        for idx in range(ep['start_idx'], ep['end_idx'] + 1):
            if idx < n:
                y_true[idx] = ep['episode_id']
    return data['artifacts'], y_true, data['episodes'], data['meta']

tune_arts_all, y_true_tune_all, tune_eps_all, tune_meta = load_split(f'{DATA_DIR}/group_gold_tune.json')
test_arts,     y_true_test,     test_eps,     test_meta  = load_split(f'{DATA_DIR}/group_gold_test.json')

n_tune      = len(tune_arts_all)
n_split     = int(n_tune * TUNE_SPLIT)

arts_all    = all_artifacts[:n_tune]
arts_early  = arts_all[:n_split]
emb_early   = all_embeddings[:n_split]
y_early     = y_true_tune_all[:n_split]

n_early_eps = len(set(y for y in y_early if y is not None))

print(f'Tune total  : {n_tune} msgs · {len(tune_eps_all)} épisodes · {tune_meta["period"]}')
print(f'Tune early  : {n_split} msgs · ~{n_early_eps} épisodes  → boundary detector')
print(f'Tune late   : {n_tune - n_split} msgs                   → Optuna (01_eval_ari.ipynb)')
print(f'Test        : {len(test_arts)} msgs · {len(test_eps)} épisodes · {test_meta["period"]}')

In [ ]:
# ── CELLULE 6 : Extraction features + labels (tune_early uniquement) ────────
from boundary_detector import extract_features, extract_labels

X_early = extract_features(emb_early, arts_early)
y_early_bin = extract_labels(y_early, len(arts_early))

n_pos = int(y_early_bin.sum())
n_neg = len(y_early_bin) - n_pos

print(f'Features : {X_early.shape}  (attendu : (n, 770))')
print(f'Labels   : {n_pos} frontières · {n_neg} continuations')
print(f'Ratio    : 1:{n_neg // max(n_pos, 1)} → pos_weight={n_neg // max(n_pos, 1)}')

In [ ]:
# ── CELLULE 7 : Entraînement BoundaryMLP (sur tune_early) ─────────────────
from boundary_detector import BoundaryDetector

detector = BoundaryDetector(device=device)
detector.fit(X_early, y_early_bin, n_epochs=30, lr=1e-3, batch_size=512, verbose=True)

print('\n✓ Entraînement terminé (sur tune_early uniquement)')

In [ ]:
# ── CELLULE 8 : Optimisation seuil sur tune_early (recall ≥ 0.90) ──────────
# On optimise le seuil sur tune_early — les données d'entraînement du detector.
# tune_late reste vierge pour Optuna dans 01_eval_ari.ipynb.
MIN_RECALL = 0.90

probs_early = detector.predict_proba(X_early)

best_f1, best_thr = 0.0, 0.5
for thr in np.arange(0.05, 0.95, 0.01):
    preds = (probs_early >= thr).astype(int)
    tp    = int(((preds == 1) & (y_early_bin == 1)).sum())
    fp    = int(((preds == 1) & (y_early_bin == 0)).sum())
    fn    = int(((preds == 0) & (y_early_bin == 1)).sum())
    prec  = tp / (tp + fp + 1e-8)
    rec   = tp / (tp + fn + 1e-8)
    f1    = 2 * prec * rec / (prec + rec + 1e-8)
    if f1 > best_f1:
        best_f1, best_thr = f1, float(thr)

# Vérifier recall >= MIN_RECALL
preds_best = (probs_early >= best_thr).astype(int)
tp = int(((preds_best == 1) & (y_early_bin == 1)).sum())
fn = int(((preds_best == 0) & (y_early_bin == 1)).sum())
if tp / (tp + fn + 1e-8) < MIN_RECALL:
    for thr in np.arange(best_thr, 0.01, -0.01):
        preds = (probs_early >= thr).astype(int)
        tp    = int(((preds == 1) & (y_early_bin == 1)).sum())
        fn    = int(((preds == 0) & (y_early_bin == 1)).sum())
        if tp / (tp + fn + 1e-8) >= MIN_RECALL:
            best_thr = float(thr)
            break
    print(f'⚠️  Seuil abaissé pour recall≥{MIN_RECALL}')

detector.threshold = best_thr

preds = (probs_early >= best_thr).astype(int)
tp    = int(((preds == 1) & (y_early_bin == 1)).sum())
fp    = int(((preds == 1) & (y_early_bin == 0)).sum())
fn    = int(((preds == 0) & (y_early_bin == 1)).sum())
prec  = tp / (tp + fp + 1e-8)
rec   = tp / (tp + fn + 1e-8)
f1    = 2 * prec * rec / (prec + rec + 1e-8)
n_b   = int(preds.sum())

print(f'Seuil retenu   : {best_thr:.2f}')
print(f'Précision      : {prec:.4f}')
print(f'Rappel         : {rec:.4f}  (cible ≥ {MIN_RECALL})')
print(f'F1             : {f1:.4f}')
print(f'Frontières     : {n_b} / {int(y_early_bin.sum())} gold sur tune_early')
print(f'Réduction      : {len(preds)/max(n_b,1):.1f}x moins d\'appels AttachScore')

In [ ]:
# ── CELLULE 9 : Évaluation sur TEST (une seule fois) ──────────────────────
# Ne pas re-tuner après avoir vu ce score

import numpy as np

probs_test = detector.predict_proba(X_test)
preds_test = (probs_test >= detector.threshold).astype(int)

tp = int(((preds_test == 1) & (y_test == 1)).sum())
fp = int(((preds_test == 1) & (y_test == 0)).sum())
fn = int(((preds_test == 0) & (y_test == 1)).sum())
prec = tp / (tp + fp + 1e-8)
rec  = tp / (tp + fn + 1e-8)
f1   = 2 * prec * rec / (prec + rec + 1e-8)

n_boundaries_test = preds_test.sum()
n_total_test = len(preds_test)

print(f"""
╔══════════════════════════════════════════════════╗
║  BOUNDARY DETECTOR — Résultat TEST               ║
╠══════════════════════════════════════════════════╣
║  Précision  : {prec:.4f}                         ║
║  Rappel     : {rec:.4f}                         ║
║  F1         : {f1:.4f}                          ║
╠══════════════════════════════════════════════════╣
║  Frontières : {n_boundaries_test} / {int(y_test.sum())} gold (sur {n_total_test} msgs)    ║
║  Taux       : {n_boundaries_test/n_total_test:.1%} des messages sont frontières    ║
║  Réduction  : {n_total_test/max(n_boundaries_test,1):.1f}x moins d'appels AttachScore       ║
╚══════════════════════════════════════════════════╝
""")

In [ ]:
# ── CELLULE 10 : Sauvegarde (local + Drive) ────────────────────────────────
LOCAL_PATH = f'{DATA_DIR}/boundary_detector.pt'
DRIVE_PATH = f'{DRIVE_DIR}/boundary_detector.pt'

detector.save(LOCAL_PATH)

# Copie vers Drive pour persister entre les sessions
import shutil
shutil.copy2(LOCAL_PATH, DRIVE_PATH)
print(f'✓ Modèle copié vers Drive → {DRIVE_PATH}')
print(f'  Seuil : {detector.threshold:.2f}')
print()
print('Étape suivante : ouvrir 01_eval_ari.ipynb et activer USE_HYBRID = True')